In [26]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)                # Force le scrollbar horizontal
pd.set_option('display.expand_frame_repr', True)


class SalesPrep():
    def __init__(self, sales_path : str):
        self.sales_path = sales_path

        self._con = None

        self._v_sales_model_base = None


    @property
    def con(self):
        if(self._con is None):
            self._con = duckdb.default_connection()

        return self._con
    
        
    @property
    def v_sales_model_base(self):
        if(self._v_sales_model_base is None):
            self._v_sales_model_base = self.view_df("v_sales_model_base")
        return self._v_sales_model_base


    def view_exists(self, view_name : str):
        exists = self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE table_name = '{view_name}'
        """).fetchone()[0]

        return exists
    

    def create_if_not_exists_view(self, view_name : str):
        if(not self.view_exists(view_name)):
            getattr(self, f"create_or_replace_view_{view_name}")()
    
    
    def create_if_not_exists_views(self, view_names : list[str]):
        for view_name in view_names:
            self.create_if_not_exists_view(view_name)


    def view_df(self, view_name : str):
            self.create_if_not_exists_view(view_name)
            df = self.con.sql(f"SELECT * FROM {view_name}").df()
            return df


    def drop_if_exists_view(self, view_name : str):
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def create_or_replace_view_v_sales(self):
        # données des ventes brutes
        v_sales = pd.read_excel(self.sales_path)
        self.con.register('v_sales', v_sales)


    def create_or_replace_view_v_sales_model(self):
        self.create_if_not_exists_view("v_sales")

        # données des ventes utilisées pour la modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model AS
            (
                SELECT 
                    * 
                FROM 
                    v_sales 
                WHERE 
                    (YEAR(DATE) < (SELECT YEAR(MAX(DATE)) FROM v_sales))
                    AND 
                    (STATUT = 'DONE')
                    AND 
                    (QUANTITE > 0)
                    AND
                    (TYPE IS NOT NULL)
                    AND
                    (GAMME IS NOT NULL)
                    AND
                    (NOM_PRODUIT IS NOT NULL)
                    
            );
        """)

        # v_sales_model = con.sql("SELECT * FROM v_sales_model").to_df()
        # print(v_sales_model.shape)
        # display(v_sales_model.head(3))


    def create_or_replace_view_v_refs(self):
        self.create_if_not_exists_view("v_sales_model")

        # nombre de mois d'activités de ventes par hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_refs AS (
            SELECT DISTINCT
                HOTEL_CODE AS hotel_code,
                NOM_BOUTIQUE AS hotel_name,
                SOLUTION AS solution,
                METRES_LINEAIRES AS metres_lineaires
            FROM 
                v_sales_model
            ORDER BY 
                hotel_code,
                hotel_name,
                solution,
                metres_lineaires
        );
        """)

        # v_nombre_mois = con.sql("SELECT * FROM v_nombre_mois").to_df()
        # print(v_nombre_mois.shape)
        # display(v_nombre_mois.head(3))


    def create_or_replace_view_v_nombre_mois(self):
        self.create_if_not_exists_view("v_sales_model")

        # nombre de mois d'activités de ventes par hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_nombre_mois AS (
            SELECT
                HOTEL_CODE AS hotel_code,
                DATE_DIFF('month', MIN(DATE), MAX(DATE)) + 1 AS nombre_mois
            FROM 
                v_sales_model
            GROUP BY
                HOTEL_CODE
            ORDER BY 
                hotel_code
        );
        """)

        # v_nombre_mois = con.sql("SELECT * FROM v_nombre_mois").to_df()
        # print(v_nombre_mois.shape)
        # display(v_nombre_mois.head(3))


    


    def create_or_replace_view_v_produit_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model"])
                    
        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme et produit
        self.con.sql("""
            CREATE OR REPLACE VIEW v_produit_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,
                TYPE as type,
                GAMME AS gamme,
                NOM_PRODUIT AS produit,

                SUM(QUANTITE) AS produit_nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS produit_nombre_paniers,

                SUM(PRIX_TTC) AS produit_montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS produit_montant_achats,
                SUM(MARGE) AS produit_montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS produit_montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS produit_montant_achats_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS produit_montant_marge_par_vente,

                
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS produit_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS produit_montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS produit_montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS produit_montant_marge_par_panier

            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                TYPE,
                GAMME,
                NOM_PRODUIT
            ORDER BY 
                hotel_code,
                type,
                gamme,
                produit
            );
        """)


    def create_or_replace_view_v_produit_paniers_ventes_mois(self):
        self.create_if_not_exists_views(["v_produit_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme et produit par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_produit_paniers_ventes_mois AS (
            SELECT 
                v_produit_paniers_ventes.*,
                nombre_mois,

                produit_nombre_ventes / nombre_mois AS produit_nombre_ventes_par_mois,
                produit_nombre_paniers / nombre_mois AS produit_nombre_paniers_par_mois,

                produit_montant_ventes / nombre_mois AS produit_montant_ventes_par_mois,
                produit_montant_achats / nombre_mois AS produit_montant_achats_par_mois,
                produit_montant_marge / nombre_mois AS produit_montant_marge_par_mois,
               
            FROM 
                v_produit_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_produit_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)


        # v_produit_paniers_ventes_mois = con.sql("SELECT * FROM v_produit_paniers_ventes_mois").to_df()
        # print(v_produit_paniers_ventes_mois.shape)
        # display(v_produit_paniers_ventes_mois.head(3))


    def create_or_replace_view_v_gamme_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model"])
                        
        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme
        self.con.sql("""
            CREATE OR REPLACE VIEW v_gamme_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,
                TYPE as type,
                GAMME AS gamme,
                
                SUM(QUANTITE) AS gamme_nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS gamme_nombre_paniers,
                COUNT(DISTINCT NOM_PRODUIT) AS gamme_nombre_produits,
                
                SUM(PRIX_TTC) AS gamme_montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS gamme_montant_achats,
                SUM(MARGE) AS gamme_montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS gamme_montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS gamme_montant_achats_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS gamme_montant_marge_par_vente,

                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS gamme_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS gamme_montant_marge_par_panier,

                SUM(QUANTITE) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_nombre_ventes_par_produit,
                SUM(PRIX_TTC) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_montant_ventes_par_produit,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_montant_achats_par_produit,
                SUM(MARGE) / COUNT(DISTINCT NOM_PRODUIT) AS gamme_montant_marge_par_produit

            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                TYPE,
                GAMME
            ORDER BY 
                hotel_code,
                type,
                gamme
            );
        """)


    def create_or_replace_view_v_gamme_paniers_ventes_mois(self):
        self.create_if_not_exists_views(["v_gamme_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) et gamme par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_gamme_paniers_ventes_mois AS (
            SELECT 
                v_gamme_paniers_ventes.*,
                nombre_mois,

                gamme_nombre_ventes / nombre_mois AS gamme_nombre_ventes_par_mois,
                gamme_nombre_paniers / nombre_mois AS gamme_nombre_paniers_par_mois,

                gamme_montant_ventes / nombre_mois AS gamme_montant_ventes_par_mois,
                gamme_montant_achats / nombre_mois AS gamme_montant_achats_par_mois,
                gamme_montant_marge / nombre_mois AS gamme_montant_marge_par_mois

            FROM 
                v_gamme_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_gamme_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)
    
        # v_gamme_paniers_ventes_mois = con.sql("SELECT * FROM v_gamme_paniers_ventes_mois").to_df()
        # print(v_gamme_paniers_ventes_mois.shape)
        # display(v_gamme_paniers_ventes_mois.head(3))



    def create_or_replace_view_v_type_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B)
        self.con.sql("""
            CREATE OR REPLACE VIEW v_type_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,
                TYPE AS type,

                SUM(QUANTITE) AS type_nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS type_nombre_paniers,
                COUNT(DISTINCT NOM_PRODUIT) AS type_nombre_produits,
                COUNT(DISTINCT GAMME) AS type_nombre_gammes,


                SUM(PRIX_TTC) AS type_montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS type_montant_achats,
                SUM(MARGE) AS type_montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS type_montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS type_montant_achats_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS type_montant_marge_par_vente,

                
                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS type_nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS type_montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS type_montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS type_montant_marge_par_panier,


                SUM(QUANTITE) / COUNT(DISTINCT NOM_PRODUIT) AS type_nombre_ventes_par_produit,
                SUM(PRIX_TTC) / COUNT(DISTINCT NOM_PRODUIT) AS type_montant_ventes_par_produit,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT NOM_PRODUIT) AS type_montant_achats_par_produit,
                SUM(MARGE) / COUNT(DISTINCT NOM_PRODUIT) AS type_montant_marge_par_produit,


                SUM(QUANTITE) / COUNT(DISTINCT GAMME) AS type_nombre_ventes_par_gamme,
                SUM(PRIX_TTC) / COUNT(DISTINCT GAMME) AS type_montant_ventes_par_gamme,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT GAMME) AS type_montant_achats_par_gamme,
                SUM(MARGE) / COUNT(DISTINCT GAMME) AS type_montant_marge_par_gamme


            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE,
                TYPE
            ORDER BY 
                hotel_code,
                type
            );
        """)
    
    
    def create_or_replace_view_v_type_paniers_ventes_mois(self):
        self.create_if_not_exists_views(["v_type_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel et type (F&B et Non F&B) par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_type_paniers_ventes_mois AS (
            SELECT 
                v_type_paniers_ventes.*,
                nombre_mois,

                type_nombre_ventes / nombre_mois AS type_nombre_ventes_par_mois,
                type_nombre_paniers / nombre_mois AS type_nombre_paniers_par_mois,

                type_montant_ventes / nombre_mois AS type_montant_ventes_par_mois,
                type_montant_achats / nombre_mois AS type_montant_achats_par_mois,
                type_montant_marge / nombre_mois AS type_montant_marge_par_mois

            FROM 
                v_type_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_type_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)

        # v_type_paniers_ventes_mois = con.sql("SELECT * FROM v_type_paniers_ventes_mois").to_df()
        # print(v_type_paniers_ventes_mois.shape)
        # display(v_type_paniers_ventes_mois.head(3))
        
    
    
    def create_or_replace_view_v_paniers_ventes(self):
        self.create_if_not_exists_views(["v_sales_model"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel
        self.con.sql("""
            CREATE OR REPLACE VIEW v_paniers_ventes AS (
            SELECT 
                HOTEL_CODE AS hotel_code,

                SUM(QUANTITE) AS nombre_ventes,
                COUNT(DISTINCT ORDER_ID) AS nombre_paniers,
                COUNT(DISTINCT NOM_PRODUIT) AS nombre_produits,
                COUNT(DISTINCT GAMME) AS nombre_gammes,
                COUNT(DISTINCT TYPE) AS nombre_types,

                SUM(PRIX_TTC) AS montant_ventes,
                SUM(PRIX_TTC_MARCHE) AS montant_acahts,
                SUM(MARGE) AS montant_marge,

                SUM(PRIX_TTC) / SUM(QUANTITE) AS montant_par_vente,
                SUM(PRIX_TTC_MARCHE) / SUM(QUANTITE) AS montant_achat_par_vente,
                SUM(MARGE) / SUM(QUANTITE) AS montant_marge_par_vente,

                

                SUM(QUANTITE) / COUNT(DISTINCT ORDER_ID) AS nombre_ventes_par_panier,
                SUM(PRIX_TTC) / COUNT(DISTINCT ORDER_ID) AS montant_ventes_par_panier,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT ORDER_ID) AS montant_achats_par_panier,
                SUM(MARGE) / COUNT(DISTINCT ORDER_ID) AS montant_marge_par_panier,


                SUM(QUANTITE) / COUNT(DISTINCT NOM_PRODUIT) AS nombre_ventes_par_produit,
                SUM(PRIX_TTC) / COUNT(DISTINCT NOM_PRODUIT) AS montant_ventes_par_produit,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT NOM_PRODUIT) AS montant_achats_par_produit,
                SUM(MARGE) / COUNT(DISTINCT NOM_PRODUIT) AS montant_marge_par_produit,


                SUM(QUANTITE) / COUNT(DISTINCT GAMME) AS nombre_ventes_par_gamme,
                SUM(PRIX_TTC) / COUNT(DISTINCT GAMME) AS montant_ventes_par_gamme,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT GAMME) AS montant_achats_par_gamme,
                SUM(MARGE) / COUNT(DISTINCT GAMME) AS montant_marge_par_gamme,

                
                SUM(QUANTITE) / COUNT(DISTINCT TYPE) AS nombre_ventes_par_type,
                SUM(PRIX_TTC) / COUNT(DISTINCT TYPE) AS montant_ventes_par_type,
                SUM(PRIX_TTC_MARCHE) / COUNT(DISTINCT TYPE) AS montant_achats_par_type,
                SUM(MARGE) / COUNT(DISTINCT TYPE) AS montant_marge_par_type
            
            FROM
                v_sales_model
            GROUP BY
                HOTEL_CODE
            ORDER BY 
                hotel_code
            );
        """)


    def create_or_replace_view_v_paniers_ventes_mois(self):
        self.create_if_not_exists_views(["v_paniers_ventes", "v_nombre_mois"])

        # nombre et montant des ventes globales et par paniers pour chaque hotel par mois
        self.con.sql("""
            CREATE OR REPLACE VIEW v_paniers_ventes_mois AS (
            SELECT 
                v_paniers_ventes.*,
                nombre_mois,

                nombre_ventes / nombre_mois AS nombre_ventes_par_mois,
                nombre_paniers / nombre_mois AS nombre_paniers_par_mois,

                montant_ventes / nombre_mois AS montant_ventes_par_mois,
                montant_acahts / nombre_mois AS montant_acahts_par_mois,
                montant_marge / nombre_mois AS montant_marge_par_mois

            FROM 
                v_paniers_ventes
            LEFT JOIN
                v_nombre_mois
            ON
                v_paniers_ventes.hotel_code= v_nombre_mois.hotel_code
            );
        """)

        # v_paniers_ventes_mois = con.sql("SELECT * FROM v_paniers_ventes_mois").to_df()
        # print(v_paniers_ventes_mois.shape)
        # display(v_paniers_ventes_mois.head(3))

            
    def create_or_replace_view_v_sales_model_base(self):
        self.create_if_not_exists_views(["v_refs", "v_produit_paniers_ventes_mois", "v_gamme_paniers_ventes_mois", "v_type_paniers_ventes_mois", "v_paniers_ventes_mois"])
                    
        # vue template pour la modélisation
        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_base AS (
                SELECT 
                    *
                FROM 
                    v_refs

                INNER JOIN
                    v_produit_paniers_ventes_mois
                USING 
                    (hotel_code)
                
                INNER JOIN 
                    v_gamme_paniers_ventes_mois
                USING 
                    (hotel_code, nombre_mois, type, gamme)
                
                INNER JOIN 
                    v_type_paniers_ventes_mois
                USING 
                    (hotel_code, nombre_mois, type)
                
                INNER JOIN 
                    v_paniers_ventes_mois
                USING 
                    (hotel_code, nombre_mois)
            );
        """)

        # v_sales_model_template = con.sql("SELECT * FROM v_sales_model_template").to_df()
        # print(v_sales_model_template.shape)
        # display(v_sales_model_template.head(3))




In [27]:
self = SalesPrep("data/hotel_sales_raw_extended_data.xlsx")

In [28]:
# self.drop_if_exists_view("v_sales")
self.view_df("v_sales").head(3)

,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


In [29]:
self.view_df("v_sales_model").head(3)

,SOLUTION,HOTEL_CODE,HOTEL_NAME,METRES_LINEAIRES,NOM_BOUTIQUE,TYPE,TYPE_RAW,GAMME,GAMME_RAW,NOM_PRODUIT,NOM_PRODUIT_RAW,CATEGORIE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE_EAN,QUANTITE,PRIX_HT,VAT,PRIX_TTC,MARQUE,FOURNISSEUR,ORDER_ID,TEMPERATURE,PRIX_TTC_MARCHE,MARGE
0,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Tongs Femme 100 Noir,TONGS FEMME 100 NOIR,NonF&B,ADIPOS,BORNE NICE,2023-08-10,12:04:22,DONE,3583787508251,1,5.000000,20.0,6.0,DECATHLON,DECATHLON,9281,25.5,6.0,0.0
1,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,PAP,ACCESSOIRES,Casquette Enfant -Mh100,CASQUETTE ENFANT -MH100,NonF&B,ADIPOS,BORNE NICE,2023-08-10,18:32:38,DONE,3608439285448,1,10.000000,20.0,12.0,DECATHLON,DECATHLON,9282,25.5,15.0,-3.0
2,simply,H2075,Ibis budget Nice Californie,6.0,Ibis budget Nice,NON-F&B,NON-F&B,ACCESSOIRES,ACCESSOIRES,Masque Easybreath de Surface Adulte - 500 Bleu,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,NonF&B,ADIPOS,BORNE NICE,2023-08-11,12:11:15,DONE,3583788165002,1,24.166667,20.0,29.0,DECATHLON,DECATHLON,9283,25.5,29.0,0.0


In [30]:
self.view_df("v_nombre_mois").head(3)

,hotel_code,nombre_mois
0,H0373,24
1,H1978,20
2,H2075,29


In [31]:
self.drop_if_exists_view("v_sales_model_base")
self._v_sales_model_base = None
self.v_sales_model_base.head(3)

BinderException: Binder Error: Ambiguous reference to column name "type" (use: "v_produit_paniers_ventes_mois.type" or "v_gamme_paniers_ventes_mois.type")

In [ ]:
self.con.sql("""
SELECT
    *,
    
FROM
    v_sales_model_base
""").to_df()

,hotel_code,hotel_name,solution,type,gamme,produit,produit_nombre_ventes,produit_nombre_paniers,produit_montant_ventes,produit_montant_achats,produit_montant_marge,produit_montant_par_vente,produit_montant_achats_par_vente,produit_montant_marge_par_vente,produit_nombre_ventes_par_panier,produit_montant_ventes_par_panier,produit_montant_achats_par_panier,produit_montant_marge_par_panier,nombre_mois,produit_nombre_ventes_par_mois,produit_nombre_paniers_par_mois,produit_montant_ventes_par_mois,produit_montant_achats_par_mois,produit_montant_marge_par_mois,gamme_nombre_ventes,gamme_nombre_paniers,gamme_nombre_produits,gamme_montant_ventes,gamme_montant_achats,gamme_montant_marge,gamme_montant_par_vente,gamme_montant_achats_par_vente,gamme_montant_marge_par_vente,gamme_nombre_ventes_par_panier,gamme_montant_ventes_par_panier,gamme_montant_achats_par_panier,gamme_montant_marge_par_panier,gamme_nombre_ventes_par_produit,gamme_montant_ventes_par_produit,gamme_montant_achats_par_produit,gamme_montant_marge_par_produit,gamme_nombre_ventes_par_mois,gamme_nombre_paniers_par_mois,gamme_montant_ventes_par_mois,gamme_montant_achats_par_mois,gamme_montant_marge_par_mois,type_nombre_ventes,type_nombre_paniers,type_nombre_produits,type_nombre_gammes,type_montant_ventes,type_montant_achats,type_montant_marge,type_montant_par_vente,type_montant_achats_par_vente,type_montant_marge_par_vente,type_nombre_ventes_par_panier,type_montant_ventes_par_panier,type_montant_achats_par_panier,type_montant_marge_par_panier,type_nombre_ventes_par_produit,type_montant_ventes_par_produit,type_montant_achats_par_produit,type_montant_marge_par_produit,type_nombre_ventes_par_gamme,type_montant_ventes_par_gamme,type_montant_achats_par_gamme,type_montant_marge_par_gamme,type_nombre_ventes_par_mois,type_nombre_paniers_par_mois,type_montant_ventes_par_mois,type_montant_achats_par_mois,type_montant_marge_par_mois,nombre_ventes,nombre_paniers,nombre_produits,nombre_gammes,nombre_types,montant_ventes,montant_acahts,montant_marge,montant_par_vente,montant_achat_par_vente,montant_marge_par_vente,nombre_ventes_par_panier,montant_ventes_par_panier,montant_achats_par_panier,montant_marge_par_panier,nombre_ventes_par_produit,montant_ventes_par_produit,montant_achats_par_produit,montant_marge_par_produit,nombre_ventes_par_gamme,montant_ventes_par_gamme,montant_achats_par_gamme,montant_marge_par_gamme,nombre_ventes_par_type,montant_ventes_par_type,montant_achats_par_type,montant_marge_par_type,nombre_ventes_par_mois,nombre_paniers_par_mois,montant_ventes_par_mois,montant_acahts_par_mois,montant_marge_par_mois
0,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,F&B,ALCOOL,Bière Duvel 33cl,14.0,14,56.0000,56.0,0.0000,4.000000,4.0,0.000000,1.000000,4.000000,4.000000,0.000000,24,0.583333,0.583333,2.333333,2.333333,0.000000,1963.0,1406,21,19795.4205,20475.0,-679.5795,10.084269,10.430464,-0.346194,1.396159,14.079246,14.562589,-0.483342,93.476190,942.639071,975.000000,-32.360929,81.791667,58.583333,824.809187,853.125,-28.315813,15541.0,8114,98,5,80937.8655,80458.0,479.8655,5.208022,5.177144,0.030877,1.915332,9.975088,9.915948,0.059140,158.581633,825.896587,821.000000,4.896587,3108.200000,16187.573100,16091.6,95.973100,647.541667,338.083333,3372.411062,3352.416667,19.994396,15759.0,8278,146,9,2,84373.3057,83953.10225,420.20345,5.353976,5.327312,0.026664,1.903721,10.192475,10.141713,0.050761,107.938356,577.899354,575.021248,2.878106,1751.000000,9374.811744,9328.122472,46.689272,7879.5,42186.65285,41976.551125,210.101725,656.625000,344.916667,3515.554404,3498.045927,17.508477
1,H0373,Mercure Paris Montmartre Sacré-Cœur,connected,F&B,ALCOOL,Bière Gallia 25cl,291.0,215,1991.0450,2037.0,-45.9550,6.842079,7.0,-0.157921,1.353488,9.260674,9.474419,-0.213744,24,12.125000,8.958333,82.960208,84.875000,-1.914792,1963.0,1406,21,19795.4205,20475.0,-679.5795,10.084269,10.430464,-0.346194,1.396159,14.079246,14.562589,-0.483342,93.476190,942.639071,975.000000,-32.360929,81.791667,58.583333,824.809187,85

In [6]:
list(self.v_sales_model_base.columns)

['hotel_code',
 'hotel_name',
 'solution',
 'type',
 'gamme',
 'produit',
 'produit_nombre_ventes',
 'produit_nombre_paniers',
 'produit_montant_ventes',
 'produit_montant_achats',
 'produit_montant_marge',
 'produit_montant_par_vente',
 'produit_montant_achats_par_vente',
 'produit_montant_marge_par_vente',
 'produit_nombre_ventes_par_panier',
 'produit_montant_ventes_par_panier',
 'produit_montant_achats_par_panier',
 'produit_montant_marge_par_panier',
 'nombre_mois',
 'produit_nombre_ventes_par_mois',
 'produit_nombre_paniers_par_mois',
 'produit_montant_ventes_par_mois',
 'produit_montant_achats_par_mois',
 'produit_montant_marge_par_mois',
 'gamme_nombre_ventes',
 'gamme_nombre_paniers',
 'gamme_nombre_produits',
 'gamme_montant_ventes',
 'gamme_montant_achats',
 'gamme_montant_marge',
 'gamme_montant_par_vente',
 'gamme_montant_achats_par_vente',
 'gamme_montant_marge_par_vente',
 'gamme_nombre_ventes_par_panier',
 'gamme_montant_ventes_par_panier',
 'gamme_montant_achats_par_pa

In [84]:
self.v_sales_model_base.to_excel("sales_model_base.xlsx", index = False)

In [ ]:
class SalesSim(SalesPrep):
    # toutes les operations sont groupe by hotel on mélange pas les données des hotels
    def __init__(self, sales_path):
        super().__init__(sales_path)

        # vue des données d'hotel pilote de simulation initialement comme le template mais ça va subir des changements
        # les modifications vont être subies uniquement par cette vue, la vue v_sales_model_base est figée
        # pour chaque scénario de simulation la vue v_sales_model_sim est créée à partir de v_sales_model_base, puis elle va subir les transformations de simulation
        self.con.sql("CREATE OR REPLACE VIEW v_sales_model_sim AS SELECT * FROM v_sales_model_base")

  
    
    def remove_produit(self, produit : str):
        pass
        # toute la suite c'est une reflexion brainstorming à la fin je me retrouve avec des incohérences donc c'est pas encore finalisé
        # quand un produit est supprimé alors théoriquement et pour chaque hotel il faut regarder les champs suivants : 
        
        # nombre_types
        # nombre_gammes
        # nombre_produits

        # type_nombre_gammes
        # type_nombre_produits

        # gamme_nombre_produits

        # nombre_paniers
        # nombre_ventes
        # montant_ventes
        
        # type_nombre_paniers
        # type_nombre_ventes
        # type_montant_ventes

        # gamme_nombre_paniers
        # gamme_nombre_ventes
        # gamme_montant_ventes
        
        

        # ces champs représentent la contribution de ce produit au nombres et aux montant totaux et par type et par gamme. 

        # on commence par les plus simples : 
        # nombre_types: on regarde le type du produit retiré, puis dans chaque hotel on regarde le nombre de produit associé à ce type,
        # s'il y en a plus que 1 alors retirer ce produit n'a pas d'impact, sinon, retirer ce produit va reduire le nombre de type par 1.

        # nombre_gammes : pareil que nombre_types, l'impact est -1 si le produit retiré etait le seul dans la gamme sinon pas d'impact.

        # nombre_produits : toujours l'impact c'est -1 car on a retiré un produit.

        # type_nombre_gammes : pareil que nombre_gamme mais appliqué au nombre de game du meme type que le produit retiré seulement
        # type_nombre_produits : pareil que nombre_produit mais appliqué au nombre de produits du meme type que le produit retiré seulement
        
        # gamme_nombre_produits : pareil que nombre_produit mais appliqué au nombre de produits du meme type et gamme que le produit retiré seulement 


        # maintenant on regarde le nombre de paniers total, l'impact est moins le nombre de paniers qui contiennent 1 seul produit qui est exactement celui retiré
        # si le panier contient un autre produit alors pas d'impact.

        # on regarde maintenant le champs nombre de ventes 
        # si on considère qu'un client n'achete qu'une fois dans l'hotel et à chaque fois n'achete qu'un produit, 
        # alors c'est équivalent au nombre de clients acheteurs.
        # on se dit que le nombre de clients acheteurs ne devrait pas changer même si on retire un produit du corner de vente. 
        # c'est juste que le nombre de ventes du produit retiré va être revéhiculé sur la liste des achats de tous les clients de l'hotel.
        # par exemple, si pour l'hotel H1, on avait P1 20, P2 40, P3 20, P4 20. et on supprime P4. alors les 20 de P4 vont être revéhiculé sur P1, P2, P3.
        # ceci donne P1  24, P2 48, P3 24 (les 20% qui voudront acheter P4 n'acheteront rien car P4 n'existe plus il y a donc une petite perte du CA mais pas total).
        
